###데이터준비

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

!wget https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
!wget https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt

train_df = pd.read_csv("ratings_train.txt", sep = "\t").dropna()
test_df = pd.read_csv("ratings_test.txt", sep = "\t").dropna()

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "test": Dataset.from_pandas(test_df)
})

print(dataset)

### Tokenizing

In [ ]:
from transformers import AutoTokenizer

# 한국어 → klue/bert-base, 영어 → bert-base-uncased
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["document"], truncation=True, padding="max_length", max_length=128)

tokenized = dataset.map(tokenize, batched=True)


### 모델 파인튜닝

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = 2)

training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 3,
    per_device_train_batch_size = 32,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized["train"],
    eval_dataset = tokenized["test"]
)

trainer.train()

### 평가 & 시각화

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

predictions = trainer.predict(tokenized["test"])
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

print(classification_report(labels, preds, target_names = ["부정", "긍정"]))
ConfusionMatrixDisplay.from_predictions(labels, preds).plot()
plt.show